# DCGM Exporter

A practical reference for **DCGM Exporter** — NVIDIA's Prometheus exporter for
GPU telemetry. It turns the rich device metrics collected by the *Data Center
GPU Manager (DCGM)* into a `/metrics` endpoint that Prometheus can scrape, so
you can monitor utilization, memory, power, temperature, throttling, and
Tensor-Core activity across a fleet of GPUs.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

### What is it?

**DCGM Exporter** (`dcgm-exporter`) is a lightweight Go service that uses the
[NVIDIA DCGM](https://developer.nvidia.com/dcgm) Go bindings to read GPU
telemetry and expose it in [Prometheus](https://prometheus.io) text format on an
HTTP endpoint (default `:9400/metrics`). It is the de-facto standard for GPU
observability on Kubernetes and is shipped as a component of the **NVIDIA GPU
Operator**.

DCGM itself is NVIDIA's suite for managing and monitoring data-center GPUs. It
runs a host engine (`nv-hostengine`) that samples hundreds of *fields* (device
attributes and counters) identified by stable IDs such as
`DCGM_FI_DEV_GPU_UTIL`. The exporter selects a subset of those fields — defined
in a CSV file — and renders each as a Prometheus gauge or counter, labeled by
GPU index, UUID, model, and (on Kubernetes) the owning pod/namespace/container.

### Why use it?

- **GPU-native metrics Prometheus understands** — utilization, framebuffer
  memory, SM/memory clocks, power draw, temperature, ECC errors, XID faults,
  PCIe/NVLink throughput, and DCGM *profiling* metrics (Tensor-Core, FP16/FP64
  pipe activity, DRAM active) that `nvidia-smi` cannot give you in time series.
- **Per-workload attribution** — on Kubernetes it maps each GPU to the pod and
  container using it via the kubelet *pod-resources* API, so you can answer
  "which job is wasting an A100?".
- **Low overhead and no in-band agent** — a single DaemonSet pod per node reads
  from the shared DCGM engine; you don't instrument your training code.
- **Ecosystem fit** — ships with the GPU Operator, has an official Grafana
  dashboard (ID `12239`), and integrates with Prometheus `ServiceMonitor`s.

### When to use it?

- You run GPU workloads on Kubernetes (training, inference, notebooks) and want
  fleet-wide utilization and health dashboards.
- You need alerting on GPU faults (XID errors, ECC double-bit errors, thermal
  throttling) before they corrupt long training runs.
- You want capacity/cost insight: are expensive GPUs actually busy, and is the
  Tensor Core engine active during training?

## Key Features

### Core Capabilities of DCGM Exporter

| Capability | Description | Why it matters |
|---|---|---|
| Field-driven metrics | Selects DCGM fields via a CSV (`*.csv`) mapping field IDs to metric names | Pick exactly the metrics you need; keep cardinality and overhead low |
| Profiling (DCP) metrics | `DCGM_FI_PROF_*` — GR engine, Tensor pipe, FP64/FP32/FP16 pipe, DRAM active, PCIe/NVLink bytes | True compute-engine occupancy, not just the coarse `GPU_UTIL` percentage |
| Kubernetes pod attribution | Joins GPU UUID to pod/namespace/container via the kubelet pod-resources socket | Per-team / per-job GPU accounting and chargeback |
| MIG awareness | Reports metrics per MIG instance with `GPU_I_ID`/`GPU_I_PROFILE` labels | Monitor sliced A100/H100 GPUs accurately |
| Health & fault signals | XID errors, ECC SBE/DBE, thermal/power throttle reasons | Catch failing hardware before it kills a run |
| Embedded or remote DCGM | Runs DCGM embedded in-process, or connects to an existing `nv-hostengine` | Avoids conflicts when DCGM is already running on the host |
| Prometheus-native output | Plain text exposition on `:9400/metrics`; `ServiceMonitor` friendly | Drops straight into existing Prometheus/Grafana stacks |

## Architecture Overview

```text
        +-------------------- GPU node --------------------+
        |                                                  |
        |   NVIDIA driver  +  NVML  +  DCGM (nv-hostengine)|
        |          ^                         ^             |
        |          | reads device fields     | DCGM API    |
        |          |                         |             |
        |   +------+-------------------------+-------+      |
        |   |            dcgm-exporter (Go)          |      |
        |   |  - loads counters CSV (field IDs)      |      |
        |   |  - maps GPU UUID -> pod (pod-resources)|      |
        |   |  - serves :9400/metrics                |      |
        |   +--------------------+-------------------+      |
        |                        | HTTP scrape             |
        +------------------------|--------------------------+
                                 v
                          Prometheus  --->  Grafana / Alertmanager
```

### Components

1. **NVIDIA driver + NVML** — the kernel driver and management library that
   actually read GPU registers.
2. **DCGM host engine (`nv-hostengine`)** — samples and caches GPU *fields*.
   The exporter can run it **embedded** (in-process, the default standalone
   behavior) or connect to a **remote/host** engine via `-r host:port`.
3. **dcgm-exporter** — selects fields from a counters CSV, collects them on an
   interval (`-c`, default `30000` ms), and renders Prometheus metrics.
4. **Counters CSV** — declarative list of `field_id, prom_type, help` rows that
   decides which metrics are exported.
5. **Pod-resources mapping (Kubernetes only)** — uses the kubelet
   `pod-resources` gRPC socket to attribute each GPU to a pod/container.
6. **Prometheus + Grafana/Alertmanager** — scrape, store, visualize, alert.

## Installation

### Prerequisites

- NVIDIA GPU with a recent **datacenter/Linux driver** installed on the host.
- **NVIDIA Container Toolkit** (`nvidia-container-toolkit`) so containers can
  access GPUs (`--gpus all` / `nvidia` runtime).
- **DCGM** available — bundled inside the exporter image, so a separate install
  is only needed if you want a shared host engine.
- For **profiling metrics** (`DCGM_FI_PROF_*`): a Volta-class GPU or newer
  (compute capability ≥ 7.0) and the `SYS_ADMIN` Linux capability on the
  container.
- On Kubernetes: the **NVIDIA GPU Operator** (recommended) or a manually
  installed device plugin + driver.

> The exporter is a **container/Go binary, not a Python/pip package**. The cell
> below shows the real ways to run it; there is nothing to `pip install`.

In [ ]:
%%bash
# DCGM Exporter is distributed as a container image, not a pip package.
# Pin to a concrete release tag in production (example tag shown).
IMAGE=nvcr.io/nvidia/k8s/dcgm-exporter:3.3.9-3.6.1-ubuntu22.04

# 1) Standalone Docker on a single GPU host.
#    --cap-add SYS_ADMIN is required for the DCGM profiling (DCP) metrics.
docker run -d --rm --gpus all --cap-add SYS_ADMIN \
  -p 9400:9400 --name dcgm-exporter "$IMAGE"

# 2) Kubernetes via Helm (standalone chart).
helm repo add gpu-helm-charts \
  https://nvidia.github.io/dcgm-exporter/helm-charts
helm repo update
helm install dcgm-exporter gpu-helm-charts/dcgm-exporter \
  --namespace gpu-monitoring --create-namespace

# 3) Recommended on K8s: let the NVIDIA GPU Operator manage it
#    (deploys driver, toolkit, device plugin, AND dcgm-exporter together).
helm install --wait gpu-operator nvidia/gpu-operator \
  --namespace gpu-operator --create-namespace


## Basic Usage

### Quick start

Once the container is running, the exporter serves Prometheus metrics on
`:9400/metrics`. Scrape it directly with `curl` to see the exposition format —
each line is `METRIC{labels} value`. Below are the most useful starter
metrics and the PromQL you'd build on top of them.

In [ ]:
%%bash
# Pull the raw exposition and show a few representative metrics.
curl -s localhost:9400/metrics | grep -E \
  'DCGM_FI_DEV_(GPU_UTIL|FB_USED|POWER_USAGE|GPU_TEMP)' | head

# Typical output (labels trimmed for readability):
#
# DCGM_FI_DEV_GPU_UTIL{gpu="0",UUID="GPU-abc...",modelName="NVIDIA A100"} 87
# DCGM_FI_DEV_FB_USED{gpu="0",UUID="GPU-abc...",modelName="NVIDIA A100"} 38942
# DCGM_FI_DEV_POWER_USAGE{gpu="0",UUID="GPU-abc..."} 312.4
# DCGM_FI_DEV_GPU_TEMP{gpu="0",UUID="GPU-abc..."} 64


In [ ]:
# Core metrics worth knowing, with example PromQL you would graph/alert on.
# (Reference table — no GPU needed to read it.)
metrics = {
    "DCGM_FI_DEV_GPU_UTIL":           "coarse % of time a kernel was running",
    "DCGM_FI_PROF_GR_ENGINE_ACTIVE":  "fraction (0-1) the graphics/compute engine was active",
    "DCGM_FI_PROF_PIPE_TENSOR_ACTIVE":"fraction the Tensor Cores were active",
    "DCGM_FI_DEV_FB_USED":            "framebuffer (VRAM) used, MiB",
    "DCGM_FI_DEV_FB_FREE":            "framebuffer free, MiB",
    "DCGM_FI_DEV_POWER_USAGE":        "instantaneous board power, watts",
    "DCGM_FI_DEV_GPU_TEMP":           "GPU die temperature, C",
    "DCGM_FI_DEV_XID_ERRORS":         "last XID error code (0 = none)",
}
for name, desc in metrics.items():
    print(f"{name:34s} {desc}")

promql = [
    # Memory utilization % per GPU.
    "100 * DCGM_FI_DEV_FB_USED / (DCGM_FI_DEV_FB_USED + DCGM_FI_DEV_FB_FREE)",
    # Average Tensor-Core activity per pod (needs pod labels from K8s).
    'avg by (exported_pod) (DCGM_FI_PROF_PIPE_TENSOR_ACTIVE)',
    # GPUs that are powered but idle for 10m (wasted capacity).
    "avg_over_time(DCGM_FI_DEV_GPU_UTIL[10m]) < 5",
]
print("\nExample PromQL:")
for q in promql:
    print("  " + q)


## Advanced Features

### 1. Custom metric selection (counters CSV)

The exporter only exports the DCGM fields listed in its counters CSV
(`/etc/dcgm-exporter/default-counters.csv` by default, overridable with `-f`).
Each row is `DCGM_FIELD_ID, prometheus_type, help text`. Trim it to what you
actually alert/graph on to reduce scrape size and collection cost; add
profiling fields when you need real engine occupancy.

### 2. Profiling (DCP) metrics

`DCGM_FI_PROF_*` fields come from DCGM's profiling subsystem and give the
ground truth on how busy the silicon is — Tensor pipe, FP64/FP32/FP16 pipe,
DRAM active, and PCIe/NVLink bytes. They require `SYS_ADMIN` and Volta+ GPUs,
and DCGM time-multiplexes some of them, so very short collection intervals can
under-sample.

### 3. Kubernetes pod attribution

When run with the kubelet `pod-resources` socket mounted (the GPU Operator does
this automatically), each metric gains `exported_pod`, `exported_namespace`,
and `exported_container` labels, enabling per-workload dashboards and chargeback.

### 4. MIG (Multi-Instance GPU)

On MIG-partitioned A100/H100 GPUs, metrics are reported per GPU *instance* with
`GPU_I_ID` and `GPU_I_PROFILE` labels so each slice is monitored independently.

In [ ]:
# A trimmed custom counters CSV: device health + profiling occupancy.
counters_csv = """# Format: DCGM_FIELD, prometheus_metric_type, help_text
DCGM_FI_DEV_GPU_UTIL,        gauge, GPU utilization (%).
DCGM_FI_DEV_FB_USED,         gauge, Framebuffer memory used (MiB).
DCGM_FI_DEV_FB_FREE,         gauge, Framebuffer memory free (MiB).
DCGM_FI_DEV_POWER_USAGE,     gauge, Power draw (W).
DCGM_FI_DEV_GPU_TEMP,        gauge, GPU temperature (C).
DCGM_FI_DEV_XID_ERRORS,      gauge, Last XID error code.
DCGM_FI_DEV_ECC_DBE_VOL_TOTAL, counter, Total volatile double-bit ECC errors.
# Profiling (DCP) metrics -- need SYS_ADMIN + Volta or newer.
DCGM_FI_PROF_GR_ENGINE_ACTIVE,   gauge, Fraction of time the GR engine is active.
DCGM_FI_PROF_PIPE_TENSOR_ACTIVE, gauge, Fraction of time the Tensor pipe is active.
DCGM_FI_PROF_DRAM_ACTIVE,        gauge, Fraction of cycles the device memory was active.
"""

with open("/tmp/custom-counters.csv", "w") as f:
    f.write(counters_csv)
print(counters_csv)

# Run the exporter against the custom field set:
#   docker run -d --gpus all --cap-add SYS_ADMIN -p 9400:9400 \
#     -v /tmp/custom-counters.csv:/etc/dcgm-exporter/custom.csv \
#     nvcr.io/nvidia/k8s/dcgm-exporter:3.3.9-3.6.1-ubuntu22.04 \
#     -f /etc/dcgm-exporter/custom.csv -c 15000


In [ ]:
# Prometheus Operator ServiceMonitor so Prometheus discovers the exporter.
service_monitor = """apiVersion: monitoring.coreos.com/v1
kind: ServiceMonitor
metadata:
  name: dcgm-exporter
  namespace: gpu-monitoring
  labels:
    release: prometheus      # must match your Prometheus serviceMonitorSelector
spec:
  selector:
    matchLabels:
      app.kubernetes.io/name: dcgm-exporter
  endpoints:
    - port: metrics          # the named :9400 port on the Service
      interval: 15s
      path: /metrics
"""
print(service_monitor)


## Use Cases

#### Cluster-wide GPU utilization & cost efficiency
- **Context:** A shared training cluster where expensive A100/H100 nodes sit
  partially idle.
- **Implementation:** Scrape `DCGM_FI_DEV_GPU_UTIL` and
  `DCGM_FI_PROF_PIPE_TENSOR_ACTIVE`; build a Grafana panel of average occupancy
  per namespace.
- **Results:** Reveals jobs requesting GPUs but barely using the Tensor Cores,
  driving right-sizing and bin-packing decisions.

#### Proactive hardware fault detection
- **Context:** Multi-day training runs that silently corrupt on bad hardware.
- **Implementation:** Alert on `DCGM_FI_DEV_XID_ERRORS != 0` and
  `DCGM_FI_DEV_ECC_DBE_VOL_TOTAL` increasing.
- **Results:** Cordon/drain the node and reschedule before the run is lost.

#### Per-team chargeback on Kubernetes
- **Context:** Multiple teams share a GPU cluster and need usage accounting.
- **Implementation:** Use the `exported_namespace`/`exported_pod` labels to sum
  GPU-seconds per team.
- **Results:** Fair-share reporting and budget enforcement.

#### Thermal / power throttling diagnosis
- **Context:** Throughput drops on hot nodes.
- **Implementation:** Track `DCGM_FI_DEV_GPU_TEMP`, `DCGM_FI_DEV_POWER_USAGE`,
  and clock-throttle reason fields.
- **Results:** Pinpoint nodes throttling due to cooling/power limits.

## Best Practices

1. **Trim the counters CSV to what you use.** Every exported field costs
   collection time and scrape bandwidth; start from the default and remove
   metrics you never query.
2. **Pin the image tag.** Use an explicit `dcgm-exporter:<dcgm>-<exporter>-<os>`
   tag, not `latest`, so the DCGM/driver compatibility is reproducible.
3. **Match the collection interval to the scrape interval.** Setting `-c` much
   shorter than Prometheus's `scrape_interval` wastes CPU; much longer makes
   gauges stale. Aligning them (e.g. both ~15s) is a good default.
4. **Mount the pod-resources socket on Kubernetes** so metrics carry pod labels;
   the GPU Operator does this for you.
5. **Grant `SYS_ADMIN` only if you need profiling metrics**, and prefer scoping
   it to the exporter pod rather than privileged mode.
6. **Use a shared/remote DCGM engine** (`-r`) if another DCGM consumer already
   runs on the host, to avoid two embedded engines fighting over the GPU.
7. **Import the official Grafana dashboard (ID 12239)** as a starting point,
   then add your fleet-specific alerts.

## Common Pitfalls

1. **Expecting `DCGM_FI_DEV_GPU_UTIL` to mean "compute busy".** It only reports
   that *a* kernel ran in the sampling window — a GPU at 100% util can still have
   near-zero Tensor-Core activity. Use `DCGM_FI_PROF_*` for real occupancy.
2. **Profiling metrics missing or zero.** Almost always a missing `SYS_ADMIN`
   capability, an unsupported (pre-Volta) GPU, or those fields not being in the
   counters CSV.
3. **No pod labels on Kubernetes.** The kubelet pod-resources socket isn't
   mounted, or you're not on the GPU Operator path — metrics show GPU index but
   no `exported_pod`.
4. **Two DCGM engines conflicting.** Running the embedded engine while
   `nv-hostengine` already runs on the host can cause errors; connect to the
   existing one with `-r localhost:5555` instead.
5. **`ServiceMonitor` not picked up.** Its labels don't match Prometheus's
   `serviceMonitorSelector`, so nothing is scraped.
6. **Driver/DCGM mismatch.** Using an exporter image whose bundled DCGM is newer
   than the host driver leads to init failures — keep them compatible.

## Performance Optimization

### Tuning for low overhead

- **`-c` / collection interval:** the dominant knob. Default is 30000 ms;
  lower it only as far as your scrape interval needs. Sub-second intervals
  greatly increase CPU and can under-sample time-multiplexed profiling fields.
- **Field count:** fewer fields in the CSV = less work per cycle and a smaller
  exposition payload. Drop NVLink/PCIe profiling fields if you don't chart them.
- **Profiling vs basic fields:** `DCGM_FI_PROF_*` are more expensive than plain
  device fields because DCGM multiplexes the profiling units; enable them
  deliberately.
- **Scrape sizing:** on dense nodes (8× GPUs, MIG slices) the metric count
  multiplies; keep Prometheus `scrape_timeout` comfortably above the exporter's
  response time.

In [ ]:
# Estimate scrape payload / cost as a function of GPUs and selected fields.
def scrape_profile(num_gpus, fields, mig_slices_per_gpu=1, interval_s=15):
    series = num_gpus * mig_slices_per_gpu * fields
    # rough exposition size: ~120 bytes/line including labels
    payload_kb = series * 120 / 1024
    samples_per_min = series * (60 / interval_s)
    return series, round(payload_kb, 1), int(samples_per_min)

for desc, (g, f, m) in {
    "1x A100, default (~60 fields)":   (1, 60, 1),
    "8x H100, trimmed (12 fields)":    (8, 12, 1),
    "8x A100 MIG 7-slice (12 fields)": (8, 12, 7),
}.items():
    s, kb, spm = scrape_profile(g, f, m)
    print(f"{desc:34s} series={s:5d}  payload~{kb:6}KB  samples/min~{spm}")


## Production Deployment

### Kubernetes DaemonSet (one exporter per GPU node)

In production you usually let the **GPU Operator** manage this, but the manifest
below shows the essential pieces if you deploy standalone: node selection to GPU
nodes only, the `SYS_ADMIN` capability for profiling, the pod-resources mount
for pod labels, and a matching Service for scraping.

```yaml
apiVersion: apps/v1
kind: DaemonSet
metadata:
  name: dcgm-exporter
  namespace: gpu-monitoring
spec:
  selector:
    matchLabels: { app.kubernetes.io/name: dcgm-exporter }
  template:
    metadata:
      labels: { app.kubernetes.io/name: dcgm-exporter }
    spec:
      nodeSelector:
        nvidia.com/gpu.present: "true"     # only GPU nodes
      tolerations:
        - key: nvidia.com/gpu
          operator: Exists
          effect: NoSchedule
      containers:
        - name: dcgm-exporter
          image: nvcr.io/nvidia/k8s/dcgm-exporter:3.3.9-3.6.1-ubuntu22.04
          args: ["-f", "/etc/dcgm-exporter/dcp-metrics-included.csv", "-c", "15000"]
          ports:
            - { name: metrics, containerPort: 9400 }
          securityContext:
            capabilities:
              add: ["SYS_ADMIN"]           # required for profiling (DCP) metrics
          volumeMounts:
            - { name: pod-resources, mountPath: /var/lib/kubelet/pod-resources }
      volumes:
        - name: pod-resources
          hostPath: { path: /var/lib/kubelet/pod-resources }
---
apiVersion: v1
kind: Service
metadata:
  name: dcgm-exporter
  namespace: gpu-monitoring
  labels: { app.kubernetes.io/name: dcgm-exporter }
spec:
  selector: { app.kubernetes.io/name: dcgm-exporter }
  ports:
    - { name: metrics, port: 9400, targetPort: 9400 }
```

## Monitoring and Observability

### Key metrics to track

- **Utilization:** `DCGM_FI_DEV_GPU_UTIL`, `DCGM_FI_PROF_GR_ENGINE_ACTIVE`,
  `DCGM_FI_PROF_PIPE_TENSOR_ACTIVE` — is the GPU (and its Tensor Cores) busy?
- **Memory:** `DCGM_FI_DEV_FB_USED` / `DCGM_FI_DEV_FB_FREE` — VRAM pressure and
  OOM risk.
- **Health:** `DCGM_FI_DEV_XID_ERRORS`, `DCGM_FI_DEV_ECC_DBE_VOL_TOTAL` — fault
  detection.
- **Thermals/power:** `DCGM_FI_DEV_GPU_TEMP`, `DCGM_FI_DEV_POWER_USAGE` plus
  clock-throttle reasons.

### Example Prometheus alerting rules

```yaml
groups:
  - name: gpu.rules
    rules:
      - alert: GPUXidError
        expr: DCGM_FI_DEV_XID_ERRORS != 0
        for: 1m
        labels: { severity: critical }
        annotations:
          summary: "XID error {{ $value }} on GPU {{ $labels.gpu }} ({{ $labels.Hostname }})"
      - alert: GPUDoubleBitECC
        expr: increase(DCGM_FI_DEV_ECC_DBE_VOL_TOTAL[10m]) > 0
        for: 0m
        labels: { severity: critical }
      - alert: GPUOverTemp
        expr: DCGM_FI_DEV_GPU_TEMP > 85
        for: 5m
        labels: { severity: warning }
      - alert: GPUIdleButAllocated
        expr: avg_over_time(DCGM_FI_DEV_GPU_UTIL[30m]) < 5
        for: 30m
        labels: { severity: info }
```

Visualize with the official **NVIDIA DCGM Exporter** Grafana dashboard
(`grafana.com` dashboard ID **12239**).

## Troubleshooting

#### `/metrics` returns no `DCGM_FI_PROF_*` lines
- **Symptoms:** Basic device metrics appear, profiling ones are absent.
- **Cause:** Missing `SYS_ADMIN` capability, a pre-Volta GPU, or those fields
  aren't in the active counters CSV.
- **Solution:** Add `--cap-add SYS_ADMIN` (or the K8s `securityContext`
  capability), confirm GPU compute capability ≥ 7.0, and include the
  `DCGM_FI_PROF_*` rows in the CSV.

#### Exporter pod crash-loops on startup
- **Symptoms:** Logs show DCGM init / "Failed to initialize NVML" errors.
- **Cause:** Driver/DCGM version mismatch, missing NVIDIA Container Toolkit, or
  no GPU visible to the container.
- **Solution:** Verify `nvidia-smi` works on the node, the `nvidia` runtime is
  configured, and the exporter image's DCGM is compatible with the host driver.

#### Metrics have no pod/namespace labels
- **Symptoms:** Only `gpu`/`UUID`/`modelName` labels, no `exported_pod`.
- **Cause:** The kubelet `pod-resources` socket isn't mounted into the pod.
- **Solution:** Mount `/var/lib/kubelet/pod-resources` (the GPU Operator does
  this) and restart the DaemonSet.

#### Prometheus shows the target as `down`
- **Symptoms:** Target unreachable or `ServiceMonitor` not discovered.
- **Cause:** Network policy blocking `:9400`, or `ServiceMonitor` labels not
  matching `serviceMonitorSelector`.
- **Solution:** `curl` the endpoint from inside the cluster and align the
  `ServiceMonitor` `release`/labels with your Prometheus selector.

## Comparison with Alternatives

| Aspect | DCGM Exporter | `nvidia-smi` / NVML scripts | node_exporter | Cloud GPU metrics (e.g. CloudWatch) |
|---|---|---|---|---|
| GPU profiling metrics | Yes (`DCGM_FI_PROF_*`) | Limited / no time series | No GPU metrics | Coarse utilization only |
| Prometheus-native | Yes (`:9400/metrics`) | Needs custom exporter | Yes (no GPU) | Provider-specific |
| Kubernetes pod attribution | Yes (pod-resources) | No | No | Rare |
| MIG support | Yes (per-instance) | Partial | No | Varies |
| Fault signals (XID/ECC) | Yes | Via parsing | No | Limited |
| Setup effort | Low (GPU Operator) | DIY scripting | n/a for GPUs | Managed but locked-in |

### When to choose DCGM Exporter

- You run GPUs on Kubernetes and already use Prometheus/Grafana.
- You need real engine/Tensor-Core occupancy and per-pod attribution, not just
  a utilization percentage.
- You want hardware fault alerting (XID/ECC) across a fleet.

Prefer a quick `nvidia-smi` loop only for ad-hoc, single-host debugging, and
rely on managed cloud metrics when you can't run a DaemonSet.

## Resources

### Official documentation
- DCGM Exporter repository: https://github.com/NVIDIA/dcgm-exporter
- NVIDIA DCGM product page: https://developer.nvidia.com/dcgm
- DCGM API / field reference: https://docs.nvidia.com/datacenter/dcgm/latest/
- NVIDIA GPU Operator docs: https://docs.nvidia.com/datacenter/cloud-native/gpu-operator/

### Tutorials and guides
- GPU telemetry with DCGM on Kubernetes:
  https://docs.nvidia.com/datacenter/cloud-native/gpu-telemetry/latest/index.html
- Official Grafana dashboard (ID 12239): https://grafana.com/grafana/dashboards/12239
- Prometheus Operator ServiceMonitor docs:
  https://prometheus-operator.dev/docs/operator/design/

### Community resources
- NVIDIA Developer Forums (DCGM): https://forums.developer.nvidia.com/c/datacenter/dcgm/
- dcgm-exporter GitHub issues: https://github.com/NVIDIA/dcgm-exporter/issues

### Related technologies
- NVIDIA GPU Operator (deploys driver, toolkit, device plugin, exporter)
- NVIDIA Device Plugin for Kubernetes
- Prometheus + Grafana + Alertmanager
- NVML / `nvidia-smi` for low-level, single-host inspection